# Borzoi test evaluation

Evaluates the final fine-tuned Borzoi model (`heads_only_deduplicated_squash`) on its test intervals:
training curves, coverage QC of the observed tracks, Pearson correlation per track over the
concatenated intervals and per interval, and gene-level RNA correlations (raw and quantile-normalised
exon counts). The per-track summary is also computed by `workflows/borzoi/03_test_eval.sh`
(streamed); this notebook holds all predictions in memory and adds the per-interval views.

**Inputs** (from `config/paths.yaml`): the test predictions `*_predictions.h5` and `metrics.csv` in
`borzoi_checkpoint_dir` (written during training), the BigWigs in `bigwig_dir` and the `gtf`.

**Memory:** P, T and their squashed versions are held in memory (about 20 GB for the full test set);
set `N_INTERVALS` to evaluate a subset.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.eval.track_correlation import (
    compare_track_coverage,
    compute_na_masked_pearson_distribution,
    density_scatter,
    plot_correlation_beeswarm,
    plot_interval_correlation_heatmap,
    plot_strand_expression_heatmaps,
    quantile_normalize_RNA_counts,
)
from cardiomics_s2f.eval.track_prediction import (
    get_counts_from_bws,
    get_gtf,
    load_predictions_from_h5,
    process_track_metadata,
    sum_exon_counts,
)
from cardiomics_s2f.eval.training_curves import plot_training_curves
from cardiomics_s2f.finetuning.borzoi.training import resolve_staged_track_paths
from cardiomics_s2f.io.data_input import SelectiveSquashTransform

%matplotlib inline

In [ ]:
paths = load_paths()
CHECKPOINT_DIR = paths.borzoi_checkpoint_dir   # final model: heads_only_deduplicated_squash

# Test predictions saved after training (exactly one per run folder)
(PREDICTIONS,) = sorted(CHECKPOINT_DIR.glob("*_predictions.h5"))
# Training log of the gReLU CSV logger (<run>/version_0/metrics.csv)
(METRICS_CSV,) = sorted(CHECKPOINT_DIR.rglob("metrics.csv"))

BIN_SIZE = 32
COVERAGE_CUTOFF_PCT = 0.10
N_INTERVALS = None   # e.g. 30 for a quick look; None = all test intervals

## Training curves

In [ ]:
train_metrics = pd.read_csv(METRICS_CSV)
fig, losses = plot_training_curves(train_metrics)
plt.show()
# fig.savefig(paths.figures_dir / "borzoi_training_curves.pdf", bbox_inches="tight")

## Prepare data for the correlation analysis

The model was trained on squashed RNA targets, so its saved predictions are on the squashed
scale (`P_sq`); `P` holds the unsquashed predictions. The observed tracks `T` are read from the
BigWigs the model was trained on and squashed the same way (`T_sq`).

In [ ]:
P, intervals_df, tracks_df = load_predictions_from_h5(PREDICTIONS)
if N_INTERVALS is not None:
    P = P[:N_INTERVALS]
    intervals_df = intervals_df.iloc[:N_INTERVALS]
P = P.astype(np.float32)
print(P.shape, intervals_df.shape, tracks_df.shape)

# BigWigs the model was trained on (staged folder names are kept in the track table)
bw_paths = resolve_staged_track_paths(tracks_df, paths.bigwig_dir)

In [ ]:
tracks_df = process_track_metadata(tracks_df)
track_names = tracks_df["id"].to_list()

tracks_df = tracks_df.set_index("id")

cnt_mask = tracks_df.Assay_type == "CnT"
atac_mask = tracks_df.Assay_type == "ATAC"
rna_mask = tracks_df.Assay_type == "RNA"

tracks_df.head(3)

In [ ]:
#ensure same cols
intervals_df = intervals_df.rename(columns = {"CHROM": "chrom",
                                             "Chrom": "chrom",
                                             "Chromosome": "chrom",
                                             "chromosome": "chrom",
                                             "Start": "start",
                                             "End": "end"})

BIN_SIZE = 32
COVERAGE_CUTOFF_PCT = 0.10
NUM_TRACKS = P.shape[1]
NUM_BINS = P.shape[2]
NUM_INTERVALS = P.shape[0]
LABEL_LEN = NUM_BINS * BIN_SIZE
SEQ_LEN = intervals_df.iloc[0]['end'] - intervals_df.iloc[0]['start']

#resize
if SEQ_LEN - LABEL_LEN != 0:
    print("Cropping Intervals...")
    CROP = (SEQ_LEN-LABEL_LEN ) // 2
    intervals_df['end'] = (intervals_df['end'] - CROP).copy()
    intervals_df['start'] = intervals_df['start'] + CROP


intervals_df.head(3)

In [ ]:
T = get_counts_from_bws(bw_paths = bw_paths["path"], coordinates_df=intervals_df, bin_size=BIN_SIZE)
T.shape

In [ ]:
rna_indices = np.flatnonzero(rna_mask.values).tolist()
borzoi_transform = SelectiveSquashTransform(
    rna_task_indices=rna_indices,
    task_axis=1,  #aka track_axis
    transform="Borzoi",
    inverse=False
)
borzoi_inverse = SelectiveSquashTransform(
    rna_task_indices=rna_indices,
    task_axis=1,
    transform="Borzoi",
    inverse=True
)

P_sq = P                      # saved predictions are already squashed
P = borzoi_inverse(P_sq)      # unsquashed predictions
T_sq = borzoi_transform(T)

### Quickly check some QC in T

In [ ]:
interval_coverage_T = np.nansum(T, axis=2)
filtered_data = [col[~np.isnan(col)] for col in interval_coverage_T.T]

plt.figure(figsize=(12*(NUM_TRACKS/30), 5))
plt.boxplot(filtered_data, flierprops=dict(marker='o', markerfacecolor='black', markeredgecolor='black', markersize=2, alpha=1))
plt.xlabel('')
plt.ylabel('Total normalized counts per Interval')
plt.xticks(ticks=np.arange(1, len(track_names) + 1), labels=track_names, rotation=45, ha='right')
plt.title('Internal coverage: Summed counts distribution across Intervals for BigWigs')

del interval_coverage_T, filtered_data

plt.show()

In [ ]:
covered_pct = (NUM_BINS - np.isnan(T).sum(axis=2)) / NUM_BINS

filtered_data = [col[~np.isnan(col)] for col in covered_pct.T]

plt.figure(figsize=(12*(NUM_TRACKS/30), 5))
plt.axhline(y=COVERAGE_CUTOFF_PCT, color='red', linewidth=0.8, zorder=0, alpha = 0.8)
plt.boxplot(filtered_data, flierprops=dict(marker='o', markerfacecolor='black', markeredgecolor='black', markersize=2, alpha=1))

plt.xlabel('')
plt.ylabel('Covered Bins per Interval')
plt.xticks(ticks=np.arange(1, len(track_names) + 1), labels=track_names, rotation=45, ha='right')
plt.title('Internal coverage: Percentage of non NaN Bins per Intervals for BigWigs')

del filtered_data

plt.show()

In [ ]:
coverage_per_track = ((NUM_INTERVALS*NUM_BINS) - np.isnan(T).sum(axis=(0, 2))) / (NUM_INTERVALS*NUM_BINS) 
coverage_per_track = pd.Series(coverage_per_track, index=track_names)

## Plot concatenated pearson correaltion
here we plot the the correaltion if we concetaten all trained intervals and treat is as one big one

In [ ]:
#compute cocnatented pearson correaltion
concat_correlations = np.zeros(NUM_TRACKS)
for t in range(NUM_TRACKS):
    T_concat = T_sq[:, t, :].ravel()
    P_concat = P_sq[:, t, :].ravel()
    
    valid_mask = ~np.isnan(T_concat) & ~np.isnan(P_concat)
    T_valid = T_concat[valid_mask]
    P_valid = P_concat[valid_mask]
    if len(T_valid) > int(NUM_INTERVALS*NUM_BINS*COVERAGE_CUTOFF_PCT):
        r = np.corrcoef(P_valid, T_valid)[0, 1]
    else:
        r = np.nan  # Assign NaN if there isn't enough valid data
        
    concat_correlations[t] = r
    del T_concat, P_concat, valid_mask, T_valid, P_valid, r 

concat_correlations = pd.Series(concat_correlations, index = track_names)

In [ ]:
na_concat_cor = concat_correlations.isna()
if na_concat_cor.any():
    na_tracks = concat_correlations[na_concat_cor].index.to_list()
    print(f"The following {len(na_tracks)} Tracks hat insufficent coverage (< {COVERAGE_CUTOFF_PCT*100}% non NaN bins) to compute a concatenated correlation:\n{na_tracks}")
    

In [ ]:
modality_concat_r = {
    "RNA": concat_correlations[rna_mask],
    "ATAC": concat_correlations[atac_mask],
    "CNT": concat_correlations[cnt_mask]
}

# Increased height from 4 to 8 to comfortably fit 2 rows of plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Maximum and Minimum Concatenated Tracks", fontsize=14, fontweight='bold')

# Convert to numpy array once for fast boolean masking
track_names_arr = np.array(track_names)

for i, (mod, r) in enumerate(modality_concat_r.items()):
    if r.empty: #ships missing modalities
            continue
    max_track_name = r.idxmax()
    max_r_val = r.loc[max_track_name]
    max_mask = (track_names_arr == max_track_name)
    T_concat_max = T[:, max_mask, :].ravel()
    P_concat_max = P[:, max_mask, :].ravel()
    valid_mask_max = ~np.isnan(T_concat_max) & ~np.isnan(P_concat_max)
    density_scatter(np.log1p(P_concat_max[valid_mask_max]), 
                    np.log1p(T_concat_max[valid_mask_max]), 
                    ax=axes[0, i], 
                    density=True, 
                    title=f"{mod} MAX: {max_track_name}\n(r = {max_r_val:.3f})")
    
    min_track_name = r.idxmin()
    min_r_val = r.loc[min_track_name]
    min_mask = (track_names_arr == min_track_name)
    T_concat_min = T[:, min_mask, :].ravel()
    P_concat_min = P[:, min_mask, :].ravel()
    valid_mask_min = ~np.isnan(T_concat_min) & ~np.isnan(P_concat_min)
    density_scatter(np.log1p(P_concat_min[valid_mask_min]), 
                    np.log1p(T_concat_min[valid_mask_min]), 
                    ax=axes[1, i], 
                    density=True, 
                    title=f"{mod} MIN: {min_track_name}\n(r = {min_r_val:.3f})")
    del T_concat_min, P_concat_min, valid_mask_min, T_concat_max, P_concat_max, valid_mask_max

plt.tight_layout()
plt.show()

del modality_concat_r, track_names_arr

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)

for ax, mask in enumerate([atac_mask, rna_mask, cnt_mask]):
    if not mask.any(): #skipt masks if we dont have tracks
            continue
    r = concat_correlations[mask]
    top_conc = r.argmax()
    compare_track_coverage(
        P_sq[:,mask,:][:,top_conc,:].ravel(),
        T_sq[:,mask,:][:,top_conc,:].ravel(), 
        ax=axes[ax],
        log_transform=False,
        title=f"Best Concat: {r.idxmax()}"
    )

plt.tight_layout(h_pad=2.0)
plt.show()

In [ ]:
plt.figure(figsize=(4, 4))

plot_df = pd.DataFrame({
    "coverage_per_track": coverage_per_track, 
    "concatenated_r": concat_correlations,
    "modality": tracks_df["Assay_type"]
})

ax = sns.scatterplot(
    data=plot_df,
    x="coverage_per_track",
    y="concatenated_r",
    hue="modality",
    zorder=3  # Ensures points sit on top of the background colors
)

x_min, x_max = -0.05, 1.05
y_min, y_max = -0.05, 1.05
x_vals = np.linspace(x_min, x_max, 100)

ax.fill_between(x_vals, x_vals, y_max, color='lightgreen', alpha=0.1, zorder=0)
ax.fill_between(x_vals, y_min, x_vals, color='lightcoral', alpha=0.1, zorder=0)
ax.plot([x_min, x_max], [x_min, x_max], color='lightgrey', lw=1, zorder=1)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

plt.axvline(x=COVERAGE_CUTOFF_PCT, color='red', linewidth=0.8, zorder=0, alpha = 0.8)

plt.xlabel("Covered (non nan) bins per track [%]")
plt.ylabel("Concatenated Pearson r per track")

plt.tight_layout()
plt.show()

del ax, plot_df

## Plot inidivual pearson correaltion
here we plotthe distribution of pearson correaltion across al 6500 test intervals

In [ ]:
print("Computing Interval Pearson Corrs...")
R_indiv = compute_na_masked_pearson_distribution(P_sq, T_sq, min_valid_pct=COVERAGE_CUTOFF_PCT, return_sum_counts=False)

In [ ]:
#as we dont need the correaltion of obs CM against predicted FB we keep only the diagonal in axis 1 and 2
R_indiv = np.diagonal(R_indiv, axis1=1, axis2=2)
# Ts = np.diagonal(Ts, axis1=1, axis2=2)
# Ps = np.diagonal(Ps, axis1=1, axis2=2)

In [ ]:
fig, ax = plot_interval_correlation_heatmap(R_indiv, intervals_df, track_names, coverage_per_track)
plt.show()

In [ ]:
filtered_data = [col[~np.isnan(col)] for col in R_indiv.T]

plt.figure(figsize=(3 + 12*(NUM_TRACKS/30), 5))

plt.boxplot(filtered_data, flierprops=dict(marker='o', markerfacecolor='black', markeredgecolor='black', markersize=2, alpha=1))

# 2. Overlay the red dots using scatter
# x-coordinates match the boxplot positions (1 to N)
x_positions = np.arange(1, len(track_names) + 1)
plt.scatter(x=x_positions, 
            y=concat_correlations,       # Replace with your actual array of values
            color='red', 
            s=50,                  # s is the size of the dot
            zorder=3,              # zorder=3 forces it to draw on top of the boxes
            label='Concat. Pear. r' # Optional: for a legend
           )

plt.xlabel('')
plt.ylabel('Pearson r')
plt.xticks(ticks=x_positions, labels=track_names, rotation=45, ha='right')
plt.title('Pearson r per Track and Interval')

# Optional: Add a legend so people know what the red dot is
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
flat_covered_pct = covered_pct.ravel()
flat_r = R_indiv.ravel()
flat_modality = np.tile(tracks_df["Assay_type"].values, NUM_INTERVALS)

plot_df = pd.DataFrame({
    "coverage_per_pair": flat_covered_pct,
    "correlation_per_pair": flat_r,
    "modality": flat_modality
}).dropna()

plt.figure(figsize=(5, 5))

ax = sns.scatterplot(
    data=plot_df,
    x="coverage_per_pair",
    y="correlation_per_pair",
    hue="modality",
    alpha=1,    # Added transparency for dense point clouds
    s=4,          # Reduced point size
    linewidth=0,  # Removed point borders to reduce visual clutter
    zorder=3
)

x_min, x_max = -0.05, 1.05
y_min, y_max = -0.05, 1.05
x_vals = np.linspace(x_min, x_max, 100)

# Background Triangles
ax.fill_between(x_vals, x_vals, y_max, color='lightgreen', alpha=0.1, zorder=0)
ax.fill_between(x_vals, y_min, x_vals, color='lightcoral', alpha=0.1, zorder=0)

# x=y Line
ax.plot([x_min, x_max], [x_min, x_max], color='lightgrey', lw=1, zorder=1)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

plt.axvline(x=COVERAGE_CUTOFF_PCT, color='red', linewidth=0.8, zorder=0, alpha = 0.8)

plt.xlabel("Covered (non-nan) fraction per interval-track")
plt.ylabel("Pearson r per interval-track")

# Make the legend markers opaque (since alpha=0.1 makes them hard to read)
leg = ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
#get median to plot in beeswarm
median_individual_cor = np.nanmedian(R_indiv, axis = 0)
median_individual_cor = pd.Series(median_individual_cor, index=track_names)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)
for ax, mask in enumerate([atac_mask, rna_mask, cnt_mask]):
    if not mask.any():
        continue

    # Mask out interval-track pairs below coverage cutoff before argmax
    r_subset = R_indiv[:, mask].copy()                     
    cov_subset = covered_pct[:, mask]                      
    r_subset[cov_subset < COVERAGE_CUTOFF_PCT] = np.nan   

    if np.all(np.isnan(r_subset)):
        print(f"WARNING: No interval-track pairs above coverage cutoff for modality ax={ax}, skipping.")
        continue

    i, t = np.unravel_index(np.nanargmax(r_subset), r_subset.shape)  # nanargmax to skip NaNs
    pos = tuple(intervals_df[['chrom', 'start', 'end']].iloc[i])
    compare_track_coverage(
        P_sq[:, mask, :][i, t, :].flatten(),
        T_sq[:, mask, :][i, t, :].flatten(),
        pos=pos, log_transform=False,
        ax=axes[ax],
        title=f"Best Interval: {tracks_df[mask].iloc[t].name}"
    )

plt.tight_layout(h_pad=2.0)
plt.show()

## gene wise pearson correaltion
here we plot hte RNA correaltion not across bins but genes (either the raw coutns or the quantile nromalized counts)

In [ ]:
GTF_FILE = paths.gtf
gtf = get_gtf(GTF_FILE, chrom_filter="autosomesXY", gene_type_filter=["protein_coding", "processed_transcript"], feature=None)

In [ ]:
exons_df = gtf[gtf["feature"] == "exon"].copy()
exons_df["interval"] = exons_df.apply(
    lambda row: (
        row["start"],     # Using the actual exon start
        row["end"]
    ), 
    axis=1
)

# group the intervals into a list for each gene_id
# This creates a pandas Series where the index is gene_id and the values are lists of intervals
exon_groups = exons_df.groupby("gene_id")["interval"].apply(list)

# Isolate the gene rows (one row per gene_id)
gtf = gtf[gtf["feature"] == "gene"].copy()

# Map the list of intervals onto the gene dataframe
gtf["exon_intervals"] = gtf["gene_id"].map(exon_groups)

#Fill any genes that had no exons with an empty list instead of NaN
gtf["exon_intervals"] = gtf["exon_intervals"].apply(lambda d: d if isinstance(d, list) else [])

genes_df = gtf.set_index("gene_name")

In [ ]:
rna_track_names = tracks_df.index[rna_mask].to_list()
rna_P = P[:, rna_mask, :]
rna_T = T[:, rna_mask, :]

In [ ]:
gene_P = sum_exon_counts(pred_array=rna_P,
                intervals_df=intervals_df,
                genes_df=genes_df,
                bin_size=BIN_SIZE,
                exon_col="exon_intervals",
                track_names=rna_track_names,
                return_log1p_counts=True,
                use_strand=True)
gene_P.head(3)

In [ ]:
gene_T = sum_exon_counts(pred_array=rna_T,
                intervals_df=intervals_df,
                genes_df=genes_df,
                bin_size=BIN_SIZE,
                exon_col="exon_intervals",
                track_names=rna_track_names,
                return_log1p_counts=True,
                use_strand=True)
gene_T.head(3)

In [ ]:
del genes_df, gtf

In [ ]:
raw_gene_cor = gene_T.corrwith(gene_P, method='pearson')

gene_norm_P = quantile_normalize_RNA_counts(gene_P)
gene_norm_T = quantile_normalize_RNA_counts(gene_T)
norm_gene_cor = gene_norm_T.corrwith(gene_norm_P, method='pearson')

# track_norm_P = quantile_normalize_RNA_counts(gene_P, center_to = "track_mean")
# track_norm_T = quantile_normalize_RNA_counts(gene_T, center_to = "track_mean")
norm_track_cor = gene_norm_T.T.corrwith(gene_norm_P.T, method='pearson')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3+(NUM_TRACKS/30)*5), sharey=True)
fig.suptitle('Gene Correlations Across Normalization Methods', fontsize=17, fontweight='bold', y=0.98)

# 1. Raw Gene Correlations
axes[0].barh(rna_track_names, raw_gene_cor, color='skyblue', edgecolor='black')
axes[0].set_title('Raw Gene Correlations', fontsize=14)
axes[0].set_xlabel('Pearson R')

# 2. Normalized Across Genes
axes[1].barh(rna_track_names, norm_gene_cor, color='lightgreen', edgecolor='black')
axes[1].set_title('Normalized Across Genes', fontsize=14)
axes[1].set_xlabel('Pearson R')

# # 3. Normalized Across Tracks
# axes[2].barh(rna_track_names, norm_track_cor, color='salmon', edgecolor='black')
# axes[2].set_title('Normalized Across Tracks', fontsize=14)
# axes[2].set_xlabel('Pearson R')

# Adjust layout to prevent clipping of the long track names on the left
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_strand_expression_heatmaps(
    gene_P=gene_P,
    gene_T=gene_T,
    gene_cors={'raw gene r': raw_gene_cor, 'norm gene r': norm_gene_cor},
    gene_cors_by_gene={'norm track r': norm_track_cor},   # gene-indexed series
    figsize=(26, 15)
)
plt.show()

In [ ]:
idx_max = {
    'raw': raw_gene_cor.idxmax(),
    'gene': norm_gene_cor.idxmax(),
    # 'track': norm_track_cor.idxmax()
}

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
fig.suptitle("Maximum & Minimum Gene Correlation Scatters", fontsize=14, fontweight='bold')

density_scatter(gene_P[idx_max['raw']], gene_T[idx_max['raw']], 
                ax=axes[0,0], title=f"Raw Gene\n{idx_max['raw']}\n(r={raw_gene_cor[idx_max['raw']]:.3f})")
density_scatter(gene_norm_P[idx_max['gene']], gene_norm_T[idx_max['gene']], 
                ax=axes[0,1], title=f"Norm Gene\n{idx_max['gene']}\n(r={norm_gene_cor[idx_max['gene']]:.3f})")
# density_scatter(track_norm_P[idx_max['track']], track_norm_T[idx_max['track']], 
#                 ax=axes[0,2], title=f"Norm across tracks cor\n{idx_max['track']}\n(r={norm_track_cor[idx_max['track']]:.3f})")

idx_min = {
    'raw': raw_gene_cor.idxmin(),
    'gene': norm_gene_cor.idxmin(),
    # 'track': norm_track_cor.idxmin()
}

density_scatter(gene_P[idx_min['raw']], gene_T[idx_min['raw']], 
                ax=axes[1,0], title=f"Raw Gene\n{idx_min['raw']}\n(r={raw_gene_cor[idx_min['raw']]:.3f})")
density_scatter(gene_norm_P[idx_min['gene']], gene_norm_T[idx_min['gene']], 
                ax=axes[1,1], title=f"Norm Gene\n{idx_min['gene']}\n(r={norm_gene_cor[idx_min['gene']]:.3f})")
# density_scatter(track_norm_P[idx_min['track']], track_norm_T[idx_min['track']], 
#                 ax=axes[1,2], title=f"Norm across tracks cor\n{idx_min['track']}\n(r={norm_track_cor[idx_min['track']]:.3f})")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={'width_ratios': [6, 3]})
plot_correlation_beeswarm([concat_correlations[rna_mask],
                           median_individual_cor[rna_mask],
                           concat_correlations[atac_mask],
                           median_individual_cor[atac_mask],
                           concat_correlations[cnt_mask],
                           median_individual_cor[cnt_mask]],
                          labels=["RNA concat", "RNA median", "ATAC concat", "ATAC median", "CnT concat", "CnT median",],
                          ax= axes[0],
                         title="Bin Resolution Correlations")

plot_correlation_beeswarm([raw_gene_cor,
                           norm_gene_cor,
                           norm_track_cor],
                          labels=["raw gene", "norm gene", "norm track"],
                          ax= axes[1],
                         title="Gene Resolution Correlations")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 4))
# plot_correlation_beeswarm([concat_correlations[rna_mask],
#                            median_individual_cor[rna_mask],
#                            concat_correlations[atac_mask],
#                            median_individual_cor[atac_mask],
#                            concat_correlations[cnt_mask],
#                            median_individual_cor[cnt_mask]],
#                           labels=["RNA concat", "RNA median", "ATAC concat", "ATAC median", "CnT concat", "CnT median",],
#                           ax= axes[0],
#                          title="Bin Resolution Correlations")

plot_correlation_beeswarm([raw_gene_cor,
                           norm_gene_cor,
                           norm_track_cor],
                          labels=["raw gene", "norm gene", "norm track"],
                          ax= axes,
                         title="Gene Resolution Correlations")

plt.tight_layout()
plt.show()

In [ ]:
#save correaltions 
tracks_df['median_individual_cor'] = median_individual_cor
tracks_df['concat_cor'] = concat_correlations
tracks_df['raw_gene_cor'] = raw_gene_cor
tracks_df['norm_gene_cor'] = norm_gene_cor
tracks_df['norm_track_cor'] = norm_track_cor
tracks_df
# Save to disk (keeping the index since it matches your track names/IDs)
# tracks_df.to_csv(paths.model_eval_dir / "Borzoi" / "tracks_correlations_notebook.csv", index=True)